# Contributors
- Javier López-Usero
- Pablo Torres

# 04. Context-Aware Recommender

## Objective
This notebook implements a **context-aware recommender** for the Instacart dataset.
Unlike previous approaches, this model does not recommend products only from global popularity or user similarity. Instead, it adapts recommendations to the **situation in which the order occurs**.

In this notebook, context is defined using temporal and behavioral variables available in the dataset:
- **Day of week** (`order_dow`)
- **Hour of day** (`order_hour_of_day`)
- **Recency** (`days_since_prior_order`)

Therefore, the recommender follows the logic of a context-aware system:

**f(User, Item, Context) -> Recommendation score**

Guiding question:

**What should we recommend given the situation?**

## Practical design
This version is intentionally RAM-optimized:
- it loads only the columns needed
- it uses compact dtypes
- it avoids unnecessary joins and giant matrices
- it builds small context tables
- it generates recommendations without running a heavy full evaluation loop



In [ ]:

import gc
import pandas as pd
import numpy as np

DATA_DIR = "."
TOP_K = 10
TOP_N_PER_CONTEXT = 200

pd.set_option("display.max_colwidth", 80)


In [ ]:
orders = pd.read_csv(
    f"{DATA_DIR}/orders.csv",
    usecols=[
        "order_id", "user_id", "eval_set", "order_number",
        "order_dow", "order_hour_of_day", "days_since_prior_order"
    ],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "eval_set": "category",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32",
    }
)

prior_items = pd.read_csv(
    f"{DATA_DIR}/order_products__prior.csv",
    usecols=["order_id", "product_id", "add_to_cart_order", "reordered"],
    dtype={
        "order_id": "int32",
        "product_id": "int32",
        "add_to_cart_order": "int16",
        "reordered": "int8",
    }
)

products = pd.read_csv(
    f"{DATA_DIR}/products.csv",
    usecols=["product_id", "product_name"],
    dtype={"product_id": "int32", "product_name": "string"}
)

print("orders:", orders.shape)
print("prior_items:", prior_items.shape)
print("products:", products.shape)


orders: (3421083, 7)
prior_items: (32434489, 4)
products: (49688, 2)


## Context construction

The context-aware component is built from three temporal and behavioral dimensions available in the Instacart structure:
- **day of week**
- **time of day**
- **recency since prior order**

These variables let the model distinguish different purchase situations. For example, the same user may need different recommendations on a weekend evening than on a weekday morning.



In [ ]:
orders_prior = orders.loc[orders["eval_set"] == "prior", [
    "order_id", "user_id", "order_number", "order_dow",
    "order_hour_of_day", "days_since_prior_order"
]].copy()

del orders
gc.collect()

prior = prior_items.merge(orders_prior, on="order_id", how="inner", copy=False)

del prior_items, orders_prior
gc.collect()

print("prior merged:", prior.shape)
prior.head()


prior merged: (32434489, 9)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,3,5,9,8.0
1,2,28985,2,1,202279,3,5,9,8.0
2,2,9327,3,0,202279,3,5,9,8.0
3,2,45918,4,1,202279,3,5,9,8.0
4,2,30035,5,0,202279,3,5,9,8.0


In [ ]:
last_order = (
    prior.groupby("user_id", observed=True)["order_number"]
    .max()
    .rename("last_order_number")
)
prior = prior.merge(last_order, on="user_id", how="left", copy=False)

train_df = prior.loc[prior["order_number"] < prior["last_order_number"]].copy()
test_df  = prior.loc[prior["order_number"] == prior["last_order_number"]].copy()

# Solo usuarios con histórico y test
valid_users = np.intersect1d(train_df["user_id"].unique(), test_df["user_id"].unique())
train_df = train_df[train_df["user_id"].isin(valid_users)].copy()
test_df = test_df[test_df["user_id"].isin(valid_users)].copy()

del prior, last_order
gc.collect()

print("train:", train_df.shape)
print("test:", test_df.shape)
print("users:", len(valid_users))


train: (30294701, 10)
test: (2139788, 10)
users: 206209


In [ ]:
DOW_MAP = {
    0: "Saturday", 1: "Sunday", 2: "Monday", 3: "Tuesday",
    4: "Wednesday", 5: "Thursday", 6: "Friday"
}

train_df["hour_bin"] = pd.cut(
    train_df["order_hour_of_day"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["night", "morning", "afternoon", "evening", "night2"],
    ordered=False,
).astype("string").replace({"night2": "night"}).astype("category")

test_df["hour_bin"] = pd.cut(
    test_df["order_hour_of_day"],
    bins=[-1, 5, 11, 16, 20, 23],
    labels=["night", "morning", "afternoon", "evening", "night2"],
    ordered=False,
).astype("string").replace({"night2": "night"}).astype("category")

train_df["recency_bin"] = np.select(
    [train_df["days_since_prior_order"].isna(),
     train_df["days_since_prior_order"] <= 7,
     train_df["days_since_prior_order"] <= 14],
    ["first_order", "frequent", "regular"],
    default="lapsed"
)
train_df["recency_bin"] = train_df["recency_bin"].astype("category")

test_df["recency_bin"] = np.select(
    [test_df["days_since_prior_order"].isna(),
     test_df["days_since_prior_order"] <= 7,
     test_df["days_since_prior_order"] <= 14],
    ["first_order", "frequent", "regular"],
    default="lapsed"
)
test_df["recency_bin"] = test_df["recency_bin"].astype("category")

train_df["dow_label"] = train_df["order_dow"].map(DOW_MAP).astype("category")
test_df["dow_label"] = test_df["order_dow"].map(DOW_MAP).astype("category")

train_df[["hour_bin", "recency_bin", "dow_label"]].head()


,hour_bin,recency_bin,dow_label
0,morning,regular,Thursday
1,morning,regular,Thursday
2,morning,regular,Thursday
3,morning,regular,Thursday
4,morning,regular,Thursday


In [ ]:
context_preview = test_df[[
    "user_id", "order_id", "order_dow", "order_hour_of_day",
    "days_since_prior_order", "hour_bin", "recency_bin"
]].copy()

context_preview["is_weekend"] = context_preview["order_dow"].isin([0, 1]).astype("int8")
context_preview["context_key"] = (
    context_preview["order_dow"].astype(str) + "_" +
    context_preview["hour_bin"].astype(str) + "_" +
    context_preview["recency_bin"].astype(str)
)

print("Context variables created successfully.")
context_preview.head(10)


Context variables created successfully.


,user_id,order_id,order_dow,order_hour_of_day,days_since_prior_order,hour_bin,recency_bin,is_weekend,context_key
219,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
220,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
221,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
222,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
223,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
224,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
225,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
226,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
227,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed
228,59897,25,6,10,25.0,morning,lapsed,0,6_morning_lapsed


## Modelling approach

Given the scale and sparsity of the Instacart dataset, a lightweight **context-aware ranking** approach was implemented.

The recommender combines three context-sensitive signals:
1. **hour-of-day popularity**: products that are strong for that time bucket
2. **day-of-week popularity**: products that are strong on that day
3. **recency popularity**: products that are strong for similar time gaps since the prior order

This keeps the model interpretable and computationally feasible while still adapting recommendations to the situation.


In [ ]:
product_name_map = dict(zip(products["product_id"].astype(int), products["product_name"].fillna("")))
del products
gc.collect()


41

In [ ]:
# score = reorder_rate * log(1 + count)

def build_context_table(df, context_col, top_n=TOP_N_PER_CONTEXT):
    grp = (
        df.groupby([context_col, "product_id"], observed=True)
          .agg(count=("product_id", "size"), reorder_rate=("reordered", "mean"))
          .reset_index()
    )
    grp["score"] = grp["reorder_rate"].astype("float32") * np.log1p(grp["count"].astype("float32"))
    grp = grp[[context_col, "product_id", "score"]]
    grp = grp.sort_values([context_col, "score"], ascending=[True, False])
    top_grp = grp.groupby(context_col, observed=True).head(top_n).copy()

    tables = {}
    for ctx, sub in top_grp.groupby(context_col, observed=True):
        tables[ctx] = dict(zip(sub["product_id"].astype(int), sub["score"].astype(float)))
    return tables

hour_scores = build_context_table(train_df, "hour_bin")
dow_scores = build_context_table(train_df, "dow_label")
recency_scores = build_context_table(train_df, "recency_bin")

print("hour contexts:", list(hour_scores.keys()))
print("dow contexts:", list(dow_scores.keys())[:3], "...")
print("recency contexts:", list(recency_scores.keys()))


hour contexts: ['afternoon', 'evening', 'morning', 'night']
dow contexts: ['Friday', 'Monday', 'Saturday'] ...
recency contexts: ['first_order', 'frequent', 'lapsed', 'regular']


In [ ]:
# Scoring weights used by the context-aware recommender
W_HOUR = 0.4
W_DOW = 0.3
W_RECENCY = 0.3

weights_summary = pd.DataFrame({
    "Signal": ["Hour of day", "Day of week", "Recency"],
    "Weight": [W_HOUR, W_DOW, W_RECENCY]
})
weights_summary


,Signal,Weight
0,Hour of day,0.4
1,Day of week,0.3
2,Recency,0.3


## Recommendation logic

For a given target context, the recommender:
1. identifies the time bucket, day of week, and recency bucket
2. retrieves products that score well in each of those context tables
3. combines the three signals into one final score
4. returns the **Top-K** products with the highest context-aware score

This means recommendations are not static: the output changes depending on the purchase situation.


In [ ]:
def recommend_context_aware(hour, day_of_week, days_since_last_order, k=TOP_K, return_scores=True):
    hour_label = (
        "morning" if 6 <= hour < 12 else
        "afternoon" if 12 <= hour < 17 else
        "evening" if 17 <= hour < 21 else
        "night"
    )
    dow_label = DOW_MAP[int(day_of_week)]
    recency_label = (
        "first_order" if pd.isna(days_since_last_order) else
        "frequent" if days_since_last_order <= 7 else
        "regular" if days_since_last_order <= 14 else
        "lapsed"
    )

    combined = {}
    for table, weight, key in [
        (hour_scores, W_HOUR, hour_label),
        (dow_scores, W_DOW, dow_label),
        (recency_scores, W_RECENCY, recency_label),
    ]:
        for pid, score in table.get(key, {}).items():
            combined[pid] = combined.get(pid, 0.0) + weight * float(score)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]
    if return_scores:
        return pd.DataFrame({
            "product_id": [pid for pid, _ in ranked],
            "product_name": [product_name_map.get(pid, "") for pid, _ in ranked],
            "score": [score for _, score in ranked],
        })
    return [pid for pid, _ in ranked]


In [ ]:

# Ex
recommend_context_aware(hour=19, day_of_week=0, days_since_last_order=6, k=10)


,product_id,product_name,score
0,24852,Banana,10.021030
1,13176,Bag of Organic Bananas,9.740297
2,21137,Organic Strawberries,8.924646
3,47209,Organic Hass Avocado,8.890436
4,27845,Organic Whole Milk,8.854983
5,21903,Organic Baby Spinach,8.723586
6,47766,Organic Avocado,8.336325
7,27966,Organic Raspberries,8.215553
8,19660,Spring Water,7.978656
9,38689,Organic Reduced Fat Milk,7.927353


In [ ]:
user_context = (
    test_df.groupby("user_id", observed=True)
           .first()[["order_hour_of_day", "order_dow", "days_since_prior_order"]]
)


def recommend_for_user(user_id, k=TOP_K):
    row = user_context.loc[user_id]
    return recommend_context_aware(
        hour=int(row["order_hour_of_day"]),
        day_of_week=int(row["order_dow"]),
        days_since_last_order=row["days_since_prior_order"],
        k=k,
        return_scores=True,
    )

sample_user = int(user_context.index[0])
print("sample_user:", sample_user)
recommend_for_user(sample_user, k=10)


sample_user: 1


,product_id,product_name,score
0,24852,Banana,9.706628
1,13176,Bag of Organic Bananas,9.181361
2,27845,Organic Whole Milk,8.395889
3,47209,Organic Hass Avocado,8.347860
4,21903,Organic Baby Spinach,8.288864
5,21137,Organic Strawberries,8.274330
6,47766,Organic Avocado,7.912389
7,27966,Organic Raspberries,7.610188
8,49235,Organic Half & Half,7.584329
9,19660,Spring Water,7.500484


In [ ]:
example_users = user_context.index[:3].tolist()

for user_id in example_users:
    print("=" * 80)
    print(f"USER {int(user_id)}")

    row = user_context.loc[user_id]
    hour_val = int(row["order_hour_of_day"])
    dow_val = int(row["order_dow"])
    recency_val = row["days_since_prior_order"]

    hour_label = (
        "morning" if 6 <= hour_val < 12 else
        "afternoon" if 12 <= hour_val < 17 else
        "evening" if 17 <= hour_val < 21 else
        "night"
    )
    dow_label = DOW_MAP[dow_val]
    recency_label = (
        "first_order" if pd.isna(recency_val) else
        "frequent" if recency_val <= 7 else
        "regular" if recency_val <= 14 else
        "lapsed"
    )

    print({
        "order_dow": dow_val,
        "order_hour_of_day": hour_val,
        "days_since_prior_order": None if pd.isna(recency_val) else float(recency_val),
        "hour_bucket": hour_label,
        "dow_label": dow_label,
        "recency_bucket": recency_label,
    })
    print("Top recommendations:")
    display(recommend_for_user(int(user_id), k=10))


USER 1
{'order_dow': 4, 'order_hour_of_day': 8, 'days_since_prior_order': 30.0, 'hour_bucket': 'morning', 'dow_label': 'Wednesday', 'recency_bucket': 'lapsed'}
Top recommendations:


,product_id,product_name,score
0,24852,Banana,9.706628
1,13176,Bag of Organic Bananas,9.181361
2,27845,Organic Whole Milk,8.395889
3,47209,Organic Hass Avocado,8.347860
4,21903,Organic Baby Spinach,8.288864
5,21137,Organic Strawberries,8.274330
6,47766,Organic Avocado,7.912389
7,27966,Organic Raspberries,7.610188
8,49235,Organic Half & Half,7.584329
9,19660,Spring Water,7.500484


USER 2
{'order_dow': 3, 'order_hour_of_day': 10, 'days_since_prior_order': 13.0, 'hour_bucket': 'morning', 'dow_label': 'Tuesday', 'recency_bucket': 'regular'}
Top recommendations:


,product_id,product_name,score
0,24852,Banana,9.960466
1,13176,Bag of Organic Bananas,9.576032
2,27845,Organic Whole Milk,8.739002
3,47209,Organic Hass Avocado,8.709528
4,21137,Organic Strawberries,8.685044
5,21903,Organic Baby Spinach,8.540816
6,47766,Organic Avocado,8.236702
7,27966,Organic Raspberries,8.075121
8,49235,Organic Half & Half,7.886741
9,38689,Organic Reduced Fat Milk,7.799855


USER 3
{'order_dow': 1, 'order_hour_of_day': 15, 'days_since_prior_order': 15.0, 'hour_bucket': 'afternoon', 'dow_label': 'Sunday', 'recency_bucket': 'lapsed'}
Top recommendations:


,product_id,product_name,score
0,24852,Banana,9.785325
1,13176,Bag of Organic Bananas,9.274404
2,27845,Organic Whole Milk,8.458096
3,47209,Organic Hass Avocado,8.427121
4,21903,Organic Baby Spinach,8.353004
5,21137,Organic Strawberries,8.339788
6,47766,Organic Avocado,8.014361
7,19660,Spring Water,7.645566
8,49235,Organic Half & Half,7.613845
9,27966,Organic Raspberries,7.606928


In [ ]:
# Final summary for handoff to 05_evaluation.ipynb
context_summary = pd.DataFrame({
    "Approach": ["Context-Aware"],
    "Context": ["Temporal (day of week, hour of day, recency)"],
    "Model Type": ["Heuristic context-aware ranking"],
    "Main Signals": ["Hour popularity + day-of-week popularity + recency popularity"],
    "Output": ["Top-10 recommendations adapted to the situation"]
})

context_summary


,Approach,Context,Model Type,Main Signals,Output
0,Context-Aware,"Temporal (day of week, hour of day, recency)",Heuristic context-aware ranking,Hour popularity + day-of-week popularity + recency popularity,Top-10 recommendations adapted to the situation
